# CSU steering — Colab runner
Thin runner: all code lives in `src/`. Loop: edit locally → commit → push → run cell 1 to pull → run a milestone cell.
Runtime → Change runtime type → GPU (L4 / A100 / T4).

In [ ]:
# 1. Get the latest code
import os
REPO = 'https://github.com/ntmphu/Predicting-Activation-Steering-Failure-from-CSU.git'
DIR = '/content/csu'
if not os.path.exists(DIR):
    !git clone -q {REPO} {DIR}
%cd {DIR}
!git pull -q && git log --oneline -1

In [ ]:
# 2. Install deps (Colab already has a CUDA torch; don't replace it)
!grep -v '^torch' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3. Hugging Face backup (private dataset repo; no Google Drive / account needed)
# Paste a HF token with WRITE access. It stays in this session only, never in git.
import os
os.environ['PYTHONUNBUFFERED'] = '1'  # show script output (progress bars) live
from getpass import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF token: ')
# restore earlier results, if any (fails harmlessly on the very first run)
!python -m src.hub pull || echo "nothing to restore yet"

In [ ]:
def backup():
    """Upload results/ to the HF repo. Run after every milestone."""
    !python -m src.hub push

## Milestones

In [ ]:
# Milestone 1: data split + prompt check
!python -m src.data --model Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
# Milestone 2a: decide the prompt format (copy the output into PROGRESS.md)
!python -m src.format_check --model Qwen/Qwen2.5-1.5B-Instruct --limit 100 --gen 30

In [ ]:
# Milestone 2b: record hidden states (after setting INSTRUCTION in src/config.py, push, re-run cell 1)
!python -m src.model --model Qwen/Qwen2.5-1.5B-Instruct
backup()

In [ ]:
# Milestone 3: steering hook checks (alpha=0 no-op, hook removal, sign)
!python -m src.steer --model Qwen/Qwen2.5-1.5B-Instruct --limit 20

In [ ]:
# Milestone 4: layer x strength sweep (~5 min on GPU), then show the plots
!python -m src.sweep --model Qwen/Qwen2.5-1.5B-Instruct --limit 100
from IPython.display import Image, display
for m in ['reply', 'all']: display(Image(f'results/figures/sweep_{m}.png'))
backup()

In [ ]:
# Milestone 4b: letter-confound diagnostic (plain vs letter-balanced vector, ~2 min)
!python -m src.diagnose --model Qwen/Qwen2.5-1.5B-Instruct --limit 100